# Boosting (sequential error correction) from scratch
Sections follow the lesson:
1. Weak learners: a depth-limited tree (stump) is only slightly better than chance
2. Reweighting intuition: the doc's A/B/C/D table, then a real 2-D picture
3. Residual fitting intuition: each new model targets what is still wrong
4. Boosting as an additive model `F_M(x) = sum_m nu * h_m(x)`
5. Two instantiations: AdaBoost (reweight) and Gradient Boosting (residuals)
6. Bias-variance perspective: boosting vs bagging vs one tree
7. Hyperparameters: n_estimators x learning_rate, depth, early stopping
8. Pitfalls: label noise and outliers

In [ ]:
import os
import warnings

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(42)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

## 1. A small weighted tree (the weak learner)
One regression-style CART tree supports sample weights (needed for reweighting boosting)
and real-valued targets (needed for residual boosting). Leaf = weighted mean of targets;
with `max_depth=1` it is a **stump**.

In [ ]:
class DecisionTree:
    """Greedy CART tree. Leaf value = (weighted) mean of targets.

    max_features: None -> consider all features at every split;
                  int  -> random subset of that size at every split (Random Forest).
    """

    def __init__(self, max_depth=None, min_samples_leaf=1, max_features=None, rng=None):
        self.max_depth = np.inf if max_depth is None else max_depth
        self.msl = min_samples_leaf
        self.max_features = max_features
        self.rng = rng if rng is not None else np.random.default_rng(0)

    def fit(self, X, y, w=None):
        w = np.ones(len(y)) if w is None else np.asarray(w, float)
        self.feat, self.thr, self.left, self.right, self.value = [], [], [], [], []
        self._grow(X, y.astype(float), w, 0)
        for name in ("feat", "thr", "left", "right", "value"):
            setattr(self, name, np.array(getattr(self, name)))
        return self

    def _new_node(self):
        for lst, v in ((self.feat, -1), (self.thr, 0.0), (self.left, -1), (self.right, -1), (self.value, 0.0)):
            lst.append(v)
        return len(self.feat) - 1

    def _grow(self, X, y, w, depth):
        node = self._new_node()
        sw = w.sum()
        self.value[node] = (w * y).sum() / sw if sw > 0 else y.mean()
        if depth >= self.max_depth or len(y) < 2 * self.msl or np.all(y == y[0]):
            return node
        j, t = self._best_split(X, y, w)
        if j is None:
            return node
        mask = X[:, j] <= t
        self.feat[node], self.thr[node] = j, t
        self.left[node] = self._grow(X[mask], y[mask], w[mask], depth + 1)
        self.right[node] = self._grow(X[~mask], y[~mask], w[~mask], depth + 1)
        return node

    def _best_split(self, X, y, w):
        n, d = X.shape
        feats = np.arange(d)
        if self.max_features is not None and self.max_features < d:
            feats = self.rng.choice(d, self.max_features, replace=False)
        Tw, Ty = w.sum(), (w * y).sum()
        best_gain, best = 1e-12, (None, None)
        parent = Ty**2 / Tw
        for j in feats:
            o = np.argsort(X[:, j], kind="stable")
            xs = X[o, j]
            cw, cy = np.cumsum(w[o])[:-1], np.cumsum((w * y)[o])[:-1]
            valid = xs[:-1] < xs[1:]
            valid[: self.msl - 1] = False
            valid[n - self.msl:] = False
            valid &= (cw > 1e-12) & (Tw - cw > 1e-12)
            if not valid.any():
                continue
            gain = np.where(valid, cy**2 / np.maximum(cw, 1e-12) + (Ty - cy) ** 2 / np.maximum(Tw - cw, 1e-12) - parent, -np.inf)
            i = int(np.argmax(gain))
            if gain[i] > best_gain:
                best_gain, best = gain[i], (j, 0.5 * (xs[i] + xs[i + 1]))
        return best

    def apply(self, X):
        """Leaf id of every row."""
        node = np.zeros(len(X), dtype=int)
        active = self.left[node] != -1
        while active.any():
            idx = np.where(active)[0]
            nd = node[idx]
            go_left = X[idx, self.feat[nd]] <= self.thr[nd]
            node[idx] = np.where(go_left, self.left[nd], self.right[nd])
            active = self.left[node] != -1
        return node

    def predict(self, X):
        return self.value[self.apply(X)]


def make_clf(n, d=10, flip=0.0, rng=rng):
    """Non-linear 2-class data: only the first 4 features matter."""
    X = rng.normal(size=(n, d))
    s = X[:, 0] + X[:, 1] * X[:, 2] + 0.8 * X[:, 3] ** 2 - 0.8
    y = (s + 0.3 * rng.normal(size=n) > 0).astype(int)
    y = np.where(rng.random(n) < flip, 1 - y, y)
    return X, y


Xtr, ytr = make_clf(600)
Xte, yte = make_clf(2000)
acc = lambda p, y: float(np.mean((p >= 0.5).astype(int) == y))

stump = DecisionTree(max_depth=1).fit(Xtr, ytr)
deep = DecisionTree().fit(Xtr, ytr)
print("stump  : train %.3f test %.3f  (weak: > 0.5 but far from perfect)" % (acc(stump.predict(Xtr), ytr), acc(stump.predict(Xte), yte)))
print("deep   : train %.3f test %.3f  (strong but high-variance)" % (acc(deep.predict(Xtr), ytr), acc(deep.predict(Xte), yte)))

## 2. Reweighting intuition
Doc table: A, B, C, D with true labels Y, N, Y, N and first-model predictions Y, N, N, Y.
C and D are wrong, so their weights go up (0.25 -> 0.35) and A, B go down (0.25 -> 0.15).
AdaBoost's exact rule does the same thing: multiply by `exp(+alpha)` if wrong, `exp(-alpha)` if right, normalise.

In [ ]:
names = np.array(list("ABCD"))
actual = np.array([1, -1, 1, -1])
pred = np.array([1, -1, -1, 1])
w = np.full(4, 0.25)
wrong = pred != actual
err = w[wrong].sum()
alpha = 0.5 * np.log((1 - err) / err)          # err = 0.5 here -> alpha = 0 (coin flip)
print("weighted error", err, "-> alpha", round(alpha, 3), "(a coin-flip learner earns no vote)")

# the doc's hand-picked update, then the exact AdaBoost rule on a 1-error variant
w_doc = np.where(wrong, 0.35, 0.15)
print("doc update:", dict(zip(names, w_doc)))
pred2 = np.array([1, -1, -1, -1])              # only C wrong -> err 0.25
w2 = np.full(4, 0.25); e2 = w2[pred2 != actual].sum(); a2 = 0.5 * np.log((1 - e2) / e2)
w2 = w2 * np.exp(-a2 * actual * pred2); w2 /= w2.sum()
print("AdaBoost update when only C is wrong: alpha=%.3f weights=%s" % (a2, np.round(w2, 3)))

Now watch it on real 2-D data: a linear-ish boundary that no single stump can draw.
Each round we fit a stump on the *weighted* data, up-weight its mistakes and repeat.

In [ ]:
def make_2d(n, rng):
    X = rng.uniform(-1, 1, size=(n, 2))
    y = (X[:, 0] ** 2 + X[:, 1] ** 2 < 0.45).astype(int) * 2 - 1       # disc: +1 inside
    return X, y


X2, y2 = make_2d(200, np.random.default_rng(5))
w = np.full(len(y2), 1 / len(y2))
snap = []                                   # (stump, weights used, errors) per round
ens = np.zeros(len(y2))
for m in range(6):
    st = DecisionTree(max_depth=1).fit(X2, (y2 + 1) / 2, w)
    h = np.where(st.predict(X2) >= 0.5, 1, -1)
    e = w[h != y2].sum(); a = 0.5 * np.log((1 - e) / max(e, 1e-12))
    snap.append((st, w.copy(), h != y2, a))
    w = w * np.exp(-a * y2 * h); w /= w.sum()
    ens += a * h

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, k in zip(axes, [0, 1, 3, 5]):
    st, wk, bad, a = snap[k]
    ax.scatter(X2[:, 0], X2[:, 1], c=np.where(y2 > 0, "tab:red", "tab:blue"), s=2 + 900 * wk, alpha=0.6, edgecolor="k", lw=0.3)
    j, t = st.feat[0], st.thr[0]
    (ax.axvline if j == 0 else ax.axhline)(t, color="k", ls="--")
    ax.set_title(f"round {k+1}: err={wk[bad].sum():.2f} alpha={a:.2f}", fontsize=9)
fig.suptitle("Marker size = sample weight: hard points grow, the next stump focuses on them")
fig.tight_layout(); fig.savefig("plots/01_reweighting_rounds.png"); plt.close(fig)
print("training accuracy of 6-stump ensemble: %.3f" % np.mean(np.sign(ens) == y2))

## 3. Residual fitting intuition
The other way to "focus on what is wrong": fit the next model to the **residuals** `y - F(x)`.
Each stage corrects the leftover error of the sum of previous stages.

In [ ]:
def make_reg(n, rng, noise=0.25):
    x = rng.uniform(0, 6, n)
    y = np.sin(x) + 0.3 * x + noise * rng.normal(size=n)
    return x[:, None], y


Xr, yr = make_reg(120, np.random.default_rng(1))
xg = np.linspace(0, 6, 300)[:, None]
F = np.full(len(yr), yr.mean()); Fg = np.full(300, yr.mean())
stages = [(Fg.copy(), yr - F)]
learners = []
for m in range(12):
    r = yr - F
    t = DecisionTree(max_depth=2).fit(Xr, r)
    learners.append(t)
    F += 0.5 * t.predict(Xr); Fg += 0.5 * t.predict(xg)
    stages.append((Fg.copy(), yr - F))
    if m < 3:
        print(f"stage {m+1}: MSE={np.mean((yr-F)**2):.4f}")

fig, axes = plt.subplots(2, 4, figsize=(15, 6), sharex=True)
for col, k in enumerate([0, 1, 3, 12]):
    Fk, rk = stages[k]
    axes[0, col].scatter(Xr[:, 0], yr, s=8, c="gray"); axes[0, col].plot(xg[:, 0], Fk, "r")
    axes[0, col].set_title("ensemble after %d trees" % k)
    axes[1, col].scatter(Xr[:, 0], rk, s=8, c="tab:blue"); axes[1, col].axhline(0, color="k", lw=0.8)
    axes[1, col].set_title("residuals y - F (shrinking)")
fig.tight_layout(); fig.savefig("plots/02_residual_fitting.png"); plt.close(fig)

## 4. Boosting as an additive model
`F_M(x) = F_0 + nu * sum_m h_m(x)`: every `h_m` is added greedily, nothing before is changed
(stagewise fitting). Prediction is just the running sum; the training loss falls with each term.

In [ ]:
def additive_predict(learners, F0, nu, X, upto=None):
    out = np.full(len(X), F0, float)
    for h in learners[:upto]:
        out += nu * h.predict(X)
    return out


# rebuild the example above with nu = 0.5 and show monotone loss decrease
F0 = yr.mean()
losses = [np.mean((yr - additive_predict(learners, F0, 0.5, Xr, m)) ** 2) for m in range(len(learners) + 1)]
print("train MSE by number of terms:", np.round(losses[:6], 3), "...", round(losses[-1], 3))
assert all(a >= b - 1e-9 for a, b in zip(losses, losses[1:]))

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(losses, "o-"); ax.set(xlabel="number of terms M", ylabel="train MSE", title="Additive model: loss falls stagewise")
fig.tight_layout(); fig.savefig("plots/03_additive_loss.png"); plt.close(fig)

## 5. Two instantiations
* **AdaBoost** (exponential loss): reweight samples, learners vote with weight `alpha`.
* **Gradient boosting** (any differentiable loss): fit pseudo-residuals (negative gradient).

Compact implementations; the dedicated lessons (14, 15) go deeper.

In [ ]:
class AdaBoost:
    def __init__(self, n_estimators=100, max_depth=1):
        self.T, self.depth = n_estimators, max_depth

    def fit(self, X, y01):
        y = 2 * y01 - 1
        w = np.full(len(y), 1 / len(y)); self.hs, self.alphas = [], []
        for _ in range(self.T):
            h = DecisionTree(max_depth=self.depth).fit(X, y01, w)
            p = np.where(h.predict(X) >= 0.5, 1, -1)
            e = np.clip(w[p != y].sum(), 1e-10, 1 - 1e-10)
            if e >= 0.5:
                break
            a = 0.5 * np.log((1 - e) / e)
            w *= np.exp(-a * y * p); w /= w.sum()
            self.hs.append(h); self.alphas.append(a)
        return self

    def score(self, X, upto=None):
        return sum(a * np.where(h.predict(X) >= 0.5, 1, -1) for a, h in list(zip(self.alphas, self.hs))[:upto])

    def predict(self, X, upto=None):
        return (self.score(X, upto) > 0).astype(int)


class GradBoostReg:
    def __init__(self, n_estimators=100, lr=0.1, max_depth=2):
        self.M, self.lr, self.depth = n_estimators, lr, max_depth

    def fit(self, X, y):
        self.F0 = y.mean(); F = np.full(len(y), self.F0); self.hs = []
        for _ in range(self.M):
            h = DecisionTree(max_depth=self.depth).fit(X, y - F)
            F += self.lr * h.predict(X); self.hs.append(h)
        return self

    def predict(self, X, upto=None):
        return additive_predict(self.hs, self.F0, self.lr, X, upto)


ada = AdaBoost(200).fit(Xtr, ytr)
print("AdaBoost (200 stumps): train %.3f test %.3f" % (np.mean(ada.predict(Xtr) == ytr), np.mean(ada.predict(Xte) == yte)))
Xrt, yrt = make_reg(1000, np.random.default_rng(2))
gb = GradBoostReg(200, 0.1, 2).fit(Xr, yr)
print("GradBoost (200 trees): test MSE %.4f (noise floor 0.0625)" % np.mean((gb.predict(Xrt) - yrt) ** 2))

## 6. Bias-variance perspective
Boosting mainly **reduces bias** (stumps are high-bias; the ensemble can be complex).
Bagging mainly **reduces variance** (deep trees are low-bias, high-variance; averaging stabilises them).

In [ ]:
def bag_predict(X, y, Xs, M=100, depth=None, seed=0):
    r = np.random.default_rng(seed); out = np.zeros((len(Xs[0]), 0)); ps = [np.zeros(len(x)) for x in Xs]
    for _ in range(M):
        i = r.integers(0, len(y), len(y))
        t = DecisionTree(max_depth=depth).fit(X[i], y[i])
        for k, x in enumerate(Xs):
            ps[k] += t.predict(x) / M
    return ps


rows = []
rows.append(("single stump", acc(stump.predict(Xtr), ytr), acc(stump.predict(Xte), yte)))
rows.append(("single deep tree", acc(deep.predict(Xtr), ytr), acc(deep.predict(Xte), yte)))
b_tr, b_te = bag_predict(Xtr, ytr, [Xtr, Xte], M=60)
rows.append(("bagging 60 deep trees", acc(b_tr, ytr), acc(b_te, yte)))
rows.append(("boosting 200 stumps", np.mean(ada.predict(Xtr) == ytr), np.mean(ada.predict(Xte) == yte)))
print("%-24s %8s %8s" % ("model", "train", "test"))
for n, a, b in rows:
    print("%-24s %8.3f %8.3f" % (n, a, b))

# test error vs rounds for AdaBoost (stumps) and (depth-3 trees)
ada3 = AdaBoost(150, max_depth=3).fit(Xtr, ytr)
rounds = np.arange(1, 151)
curve = lambda mdl, X, y: [np.mean(mdl.predict(X, upto=k) == y) for k in rounds if k <= len(mdl.hs)]
fig, ax = plt.subplots(figsize=(6.5, 4))
for mdl, lab in ((ada, "stumps"), (ada3, "depth-3")):
    ax.plot(curve(mdl, Xtr, ytr), "--", label=lab + " train"); ax.plot(curve(mdl, Xte, yte), label=lab + " test")
ax.axhline(rows[1][2], color="gray", ls=":", label="single deep tree test")
ax.set(xlabel="boosting rounds", ylabel="accuracy", title="Boosting: bias falls as rounds grow"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("plots/04_bias_variance.png"); plt.close(fig)

## 7. Hyperparameters
* `n_estimators` and `learning_rate` trade off: small `nu` needs more rounds but generalises better.
* `max_depth` sets interaction order (stumps = additive model, depth d = d-way interactions).
* Early stopping: stop when validation loss stops improving.

In [ ]:
Xa, ya = make_reg(300, np.random.default_rng(10), noise=0.5)
Xv, yv = make_reg(1000, np.random.default_rng(11), noise=0.5)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lr in (1.0, 0.3, 0.1, 0.03):
    mdl = GradBoostReg(300, lr, 2).fit(Xa, ya)
    te = [np.mean((mdl.predict(Xv, upto=k) - yv) ** 2) for k in range(0, 301, 5)]
    axes[0].plot(range(0, 301, 5), te, label=f"nu={lr}")
    print(f"nu={lr:4}: best val MSE {min(te):.4f} at round {5*int(np.argmin(te))}")
axes[0].set(xlabel="n_estimators", ylabel="validation MSE", title="learning_rate vs n_estimators", ylim=(0.2, 0.6)); axes[0].legend()
for d in (1, 2, 4, 6):
    mdl = GradBoostReg(300, 0.1, d).fit(Xa, ya)
    axes[1].plot(range(0, 301, 5), [np.mean((mdl.predict(Xv, upto=k) - yv) ** 2) for k in range(0, 301, 5)], label=f"depth={d}")
axes[1].set(xlabel="n_estimators", title="max_depth (nu=0.1)", ylim=(0.2, 0.6)); axes[1].legend()
fig.tight_layout(); fig.savefig("plots/05_hyperparameters.png"); plt.close(fig)

# early stopping
mdl = GradBoostReg(300, 0.1, 3).fit(Xa, ya)
val = np.array([np.mean((mdl.predict(Xv, upto=k) - yv) ** 2) for k in range(301)])
best = int(val.argmin()); patience, stop = 20, None
for k in range(1, 301):                      # simple patience rule
    if k - int(val[:k + 1].argmin()) >= patience:
        stop = k; break
print("early stopping (patience 20) halts at round", stop, "| true best round", best)

## 8. Pitfalls: label noise and outliers
Boosting keeps pushing on hard points; if those are *noise*, it memorises them.
AdaBoost is hit hardest (exponential loss: wrong points get exponentially large weight).

In [ ]:
rows = []
for flip in (0.0, 0.1, 0.2):
    Xn, yn = make_clf(600, flip=flip, rng=np.random.default_rng(3))
    a = AdaBoost(200).fit(Xn, yn)
    rows.append((flip, np.mean(a.predict(Xn) == yn), np.mean(a.predict(Xte) == yte)))
    print("label noise %.0f%%: train acc %.3f | clean test acc %.3f" % (100 * flip, rows[-1][1], rows[-1][2]))

# outlier effect on gradient boosting: squared loss chases a single huge outlier
xo, yo = make_reg(100, np.random.default_rng(4), noise=0.1)
yo_bad = yo.copy(); yo_bad[50] += 15
fit_clean = GradBoostReg(100, 0.2, 2).fit(xo, yo); fit_bad = GradBoostReg(100, 0.2, 2).fit(xo, yo_bad)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot([r[0] for r in rows], [r[2] for r in rows], "o-"); axes[0].set(xlabel="label-noise fraction", ylabel="clean test accuracy", title="AdaBoost vs label noise")
axes[1].scatter(xo[:, 0], yo_bad, s=8, c="gray")
axes[1].plot(xg[:, 0], fit_clean.predict(xg), label="clean data"); axes[1].plot(xg[:, 0], fit_bad.predict(xg), label="with 1 outlier")
axes[1].set(title="Squared-loss boosting chases an outlier"); axes[1].legend()
fig.tight_layout(); fig.savefig("plots/06_noise_outliers.png"); plt.close(fig)
print("max change in fit caused by 1 outlier: %.2f" % np.abs(fit_bad.predict(xg) - fit_clean.predict(xg)).max())

## Summary
* Boosting = sequential weak learners, each focusing on the errors left by the ensemble (via weights or residuals).
* Final model is an additive sum; training loss decreases monotonically.
* Reduces bias; control variance with small learning rate, shallow trees, early stopping.
* Sensitive to noisy labels/outliers, so use robust losses and regularisation.